# ChuckleNet: Simple Working Pipeline
## Uses EXISTING data with labels - no extraction needed!

**This notebook:**
1. Uses pre-extracted WavLM + Prosody from Kaggle dataset
2. Trains fusion model with ALL fixes
3. Runs in ~10 minutes

**Data:** 21,468 samples (22.7% positive) - already labeled

In [ ]:
# @title Step 1: Install dependencies
!pip install -q kaggle scikit-learn

In [ ]:
# @title Step 2: Download pre-extracted data (has LABELS)
import os
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)

# Download the dataset that has BOTH embeddings + labels
!kaggle datasets download -d subhajitdas/chuckle-wavlm-training-data -p {DATA_DIR} --unzip -q

print('✅ Data downloaded')
!ls -la

In [ ]:
# @title Step 3: Load the data
import numpy as np

# Find the npz file
from pathlib import Path
npz_file = list(Path(DATA_DIR).rglob('*.npz'))
print(f'NPZ files: {npz_file}')

if npz_file:
    data = np.load(npz_file[0])
    print(f'Keys: {list(data.keys())}')
    
    embeddings = data['embeddings']
    labels = data['labels']
    
    print(f'\nTotal samples: {len(embeddings)}')
    print(f'WavLM dim: {embeddings.shape[1]}')
    print(f'Positive: {sum(labels)} ({sum(labels)/len(labels)*100:.1f}%)')

In [ ]:
# @title Step 4: Train/Val/Test split
from sklearn.model_selection import train_test_split

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

In [ ]:
# @title Step 5: Training (ALL FIXES)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, classification_report
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# DataLoaders
train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

print(f'Loaders ready: {len(train_loader)} train batches')

In [ ]:
# @title Step 6: Model + Training
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        return self.net(x)

model = SimpleClassifier(input_dim=X_train.shape[1]).to(device)

# FIXED: Class weights [1.0, 2.5] + CrossEntropyLoss
class_weights = torch.tensor([1.0, 2.5], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for emb_b, labels_b in train_loader:
        emb_b = emb_b.to(device)
        labels_b = labels_b.to(device)
        
        optimizer.zero_grad()
        logits = model(emb_b)
        loss = criterion(logits, labels_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # FIXED
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for emb_b, labels_b in val_loader:
            emb_b = emb_b.to(device)
            logits = model(emb_b)
            preds = torch.argmax(logits, dim=-1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels_b.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    epoch_time = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, '/content/best_model.pt')
        print(f'  ✅ New best!')

In [ ]:
# @title Step 7: Final Evaluation
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for emb_b, labels_b in test_loader:
        emb_b = emb_b.to(device)
        logits = model(emb_b)
        preds = torch.argmax(logits, dim=-1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels_b.numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
print(f'\n🏆 Test F1: {test_f1:.4f}')
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))

print('\n✅ Done! Model saved to /content/best_model.pt')